# Table 2 Notebook (Best eval_hard_geo_mean)

For each run folder, this notebook picks the record with the **best `eval_hard_geo_mean`** and keeps:

- `eval_geo_mean@best_eval_hard_geo_mean`
- `best_eval_hard_geo_mean`
- `epoch`


In [ ]:
import ast
import os
import re

import matplotlib.pyplot as plt
import numpy as np

ROOT = "/mnt/nushare2/data/baliao/dpc/v2-base-optim_clean-final_fp32/fold0"
KEY = "eval_geo_mean"
HARD_KEY = "eval_hard_geo_mean"
ANNOTATE = True


In [ ]:
def collect_table2_rows(root):
    rows = []
    if not os.path.isdir(root):
        raise FileNotFoundError(f"Root directory not found: {root}")

    for name in sorted(os.listdir(root)):
        run_dir = os.path.join(root, name)
        if not os.path.isdir(run_dir):
            continue

        log_path = os.path.join(run_dir, "log.out")
        if not os.path.isfile(log_path):
            continue

        best = None
        with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if KEY not in line or HARD_KEY not in line:
                    continue
                m = re.search(r"\{.*\}", line)
                if not m:
                    continue
                try:
                    rec = ast.literal_eval(m.group(0))
                    geo = float(str(rec.get(KEY)))
                    hard_geo = float(str(rec.get(HARD_KEY)))
                    epoch = float(str(rec.get("epoch")))
                except Exception:
                    continue

                if best is None or hard_geo > best[1]:
                    best = (geo, hard_geo, epoch)

        if best is not None:
            rows.append(
                {
                    "sub_dir": name,
                    "eval_geo_mean@best_eval_hard_geo_mean": best[0],
                    "best_eval_hard_geo_mean": best[1],
                    "epoch": best[2],
                }
            )

    rows.sort(key=lambda r: r["best_eval_hard_geo_mean"], reverse=True)
    return rows


rows = collect_table2_rows(ROOT)
if not rows:
    raise ValueError(f"No valid rows found under {ROOT}")

try:
    import pandas as pd
    from IPython.display import display
    df = pd.DataFrame(rows)
    display(df)
except Exception:
    for i, r in enumerate(rows, start=1):
        print(
            f"{i:>3} {r['sub_dir']:<20} "
            f"{r['eval_geo_mean@best_eval_hard_geo_mean']:.4f} "
            f"{r['best_eval_hard_geo_mean']:.4f} "
            f"epoch={r['epoch']}"
        )


In [ ]:
x = [r["eval_geo_mean@best_eval_hard_geo_mean"] for r in rows]
y = [r["best_eval_hard_geo_mean"] for r in rows]
names = [r["sub_dir"] for r in rows]

plt.figure(figsize=(10, 7))
plt.scatter(x, y, s=55, alpha=0.9, label="points")

if len(x) >= 2:
    x_arr = np.array(x, dtype=float)
    y_arr = np.array(y, dtype=float)
    slope, intercept = np.polyfit(x_arr, y_arr, 1)
    x_fit = np.linspace(x_arr.min(), x_arr.max(), 100)
    y_fit = slope * x_fit + intercept
    plt.plot(x_fit, y_fit, color="crimson", linewidth=2, label=f"fit: y={slope:.3f}x+{intercept:.3f}")

if ANNOTATE:
    for name, xv, yv in zip(names, x, y):
        plt.annotate(name, (xv, yv), fontsize=8, xytext=(4, 4), textcoords="offset points")

plt.xlabel("eval_geo_mean@best_eval_hard_geo_mean")
plt.ylabel("best_eval_hard_geo_mean")
plt.title("best_eval_hard_geo_mean vs eval_geo_mean@best_eval_hard_geo_mean")
plt.grid(alpha=0.25, linestyle="--")
plt.legend()
plt.tight_layout()
plt.show()
